[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/quickstart.ipynb)

# Valuein US Core Fundamentals — Getting Started

The first notebook every new user should run. Confirms your token works,
loads two lightweight tables, and shows you how to look up a company
by ticker. Completes in under 30 seconds.

---

In [ ]:
!pip install valuein-sdk -q

In [ ]:
try:
    import os

    from google.colab import userdata

    os.environ["VALUEIN_API_KEY"] = userdata.get("VALUEIN_API_KEY")
except ImportError:
    pass

## Step 1: Connect and verify

`tables=[]` skips all data downloads — instant auth check.

In [ ]:
from valuein_sdk import ValueinClient

print("Connecting to Valuein gateway...")
client = ValueinClient(tables=[])

me = client.me()
snap = client.manifest().get("snapshot", "unknown")
print(f"  Plan     : {me.get('plan', 'unknown')}")
print(f"  Status   : {me.get('status', 'unknown')}")

## Step 2: Load entity + security (lightweight, no financials)

In [ ]:
print("Loading entity and security tables...")
client = ValueinClient(tables=["entity", "security"])

counts = client.query("""
    SELECT
        (SELECT count(*) FROM entity)   AS entities,
        (SELECT count(*) FROM security) AS securities
""")
print(f"  Entities  : {counts['entities'].iloc[0]:,}")
print(f"  Securities: {counts['securities'].iloc[0]:,}")

## Step 3: Look up a company by ticker

In [ ]:
print("Looking up AAPL...")
df = client.query("""
    SELECT
        e.cik,
        e.name,
        e.sector,
        e.industry,
        e.status,
        s.symbol,
        s.exchange
    FROM security s
    JOIN entity   e ON s.entity_id = e.cik
    WHERE s.symbol    = 'AAPL'
      AND s.is_active = TRUE
    LIMIT 1
""")
print(df.to_string(index=False))

print()
print("Setup complete. You're ready to query US Core Fundamentals.")

## Next steps

- [fundamental_analysis.ipynb](fundamental_analysis.ipynb) — revenue trends, balance sheets, margin ratios
- [pit_backtest.ipynb](pit_backtest.ipynb) — point-in-time backtesting and look-ahead bias
- [survivorship_bias.ipynb](survivorship_bias.ipynb) — delisted and bankrupt companies in the dataset